In [1]:
!pip install "pyautogen[retrievechat]==0.2.6" -qqq

ERROR: Could not find a version that satisfies the requirement pyautogen==0.2.6 (from versions: 0.0.1, 0.1.0, 0.1.1rc1, 0.1.1, 0.1.2, 0.1.3, 0.1.4, 0.1.5, 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.1.10, 0.1.11, 0.1.12, 0.1.13, 0.1.14, 0.2.0b1, 0.2.0b2, 0.2.7, 0.2.8, 0.2.9, 0.2.10, 0.2.11, 0.2.12, 0.2.13, 0.2.14, 0.2.15, 0.2.16, 0.2.17, 0.2.18, 0.2.19, 0.2.20, 0.2.21, 0.2.22, 0.2.23, 0.2.24, 0.2.25, 0.2.26, 0.2.27, 0.2.28, 0.2.29, 0.2.30, 0.2.31, 0.2.32, 0.2.33, 0.2.34, 0.2.35, 0.3.0b4, 0.3.0, 0.3.1, 0.3.2b1, 0.3.2b2, 0.3.2, 0.4b1, 0.4, 0.4.1, 0.4.2b1, 0.5.0b2, 0.5.0, 0.5.1, 0.5.2b1, 0.5.2, 0.5.3b1, 0.5.3, 0.6.0b1, 0.6.0b2, 0.6.0, 0.6.1, 0.7.0b1, 0.7.0b2, 0.7.0b3, 0.7.0, 0.7.1, 0.7.2b1, 0.7.2, 0.7.3, 0.7.4b1, 0.7.4b2, 0.7.4, 0.7.5, 0.7.6b1, 0.7.6, 0.8.0b1, 0.8.0, 0.8.1, 0.8.2rc0, 0.8.2)
ERROR: No matching distribution found for pyautogen==0.2.6


In [27]:
%pip install "pyautogen[retrievechat]>=0.2.6" -qqq
%pip install ag2[openai] -qqq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-callbacks-wandb 0.1.2 requires wandb<0.17.0,>=0.16.2, but you have wandb 0.19.8 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 예제 15.2 OpenAI API 키 설정

In [3]:
import json
import os

openai_api_key = os.getenv("OPENAI_API_KEY")

with open('OAI_CONFIG_LIST.json', 'w') as f:
  config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": openai_api_key
    },
    {
        "model": "gpt-4o",
        "api_key": openai_api_key,
    },
    {
        "model": "dall-e-3",
        "api_key": openai_api_key,
    }
]
  json.dump(config_list, f)

## 예제 15.3 에이전트에 사용할 설정 불러오기

In [7]:
import autogen

config_list = autogen.config_list_from_json(
    "OAI_CONFIG_LIST.json",
    file_location=".",
    filter_dict={
        "model": ["gpt-4o-mini"],
    },
)

llm_config = {
    "config_list": config_list,
    "temperature": 0,
}

code_execution_config = {
    "use_docker": False
}

## 예제 15.4 AutoGen의 핵심 구성요소인 UserProxyAgent와 AssistantAgent

In [11]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent("assistant", llm_config=llm_config)
user_proxy = UserProxyAgent("user_proxy",
  is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
  human_input_mode="NEVER",
  code_execution_config={"work_dir": "coding", "use_docker": False})

## 예제 15.5 삼성전자의 3개월 주식 가격 그래프를 그리는 작업 실행

In [4]:
user_proxy.initiate_chat(assistant, message="""
삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장하는 코드를 파일로 `samsung_stock_price.py`라는 파일로 저장한 후 실행해줘.
plotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.
값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.
이미지 비율은 보기 좋게 적절히 설정해줘.
""")

user_proxy (to assistant):


삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장하는 코드를 파일로 `samsung_stock_price.py`라는 파일로 저장한 후 실행해줘.
plotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.
값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.
이미지 비율은 보기 좋게 적절히 설정해줘.


--------------------------------------------------------------------------------
assistant (to user_proxy):

먼저, 삼성전자의 지난 3개월 주식 가격 데이터를 가져오기 위해 `yfinance` 라이브러리를 사용할 것입니다. 그 다음, `plotly` 라이브러리를 사용하여 그래프를 그린 후, 요구사항에 맞게 설정하겠습니다. 

아래는 `samsung_stock_price.py`라는 파일에 저장할 코드입니다.

```python
# filename: samsung_stock_price.py
import yfinance as yf
import plotly.graph_objects as go
from datetime import datetime, timedelta

# 삼성전자 주식 데이터 가져오기
end_date = datetime.now()
start_date = end_date - timedelta(days=90)
samsung_stock = yf.download('005930.KS', start=start_date, end=end_date)

# 그래프 그리기
fig = go.Figure()

# 주가 선 그래프 추가
fig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock['Close'], mode='lines', name='Close Price'))

# 그래프 아래를 투명한 녹색으로

ChatResult(chat_id=None, chat_history=[{'content': '\n삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장하는 코드를 파일로 `samsung_stock_price.py`라는 파일로 저장한 후 실행해줘.\nplotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.\n값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.\n이미지 비율은 보기 좋게 적절히 설정해줘.\n', 'role': 'assistant', 'name': 'user_proxy'}, {'content': '먼저, 삼성전자의 지난 3개월 주식 가격 데이터를 가져오기 위해 `yfinance` 라이브러리를 사용할 것입니다. 그 다음, `plotly` 라이브러리를 사용하여 그래프를 그린 후, 요구사항에 맞게 설정하겠습니다. \n\n아래는 `samsung_stock_price.py`라는 파일에 저장할 코드입니다.\n\n```python\n# filename: samsung_stock_price.py\nimport yfinance as yf\nimport plotly.graph_objects as go\nfrom datetime import datetime, timedelta\n\n# 삼성전자 주식 데이터 가져오기\nend_date = datetime.now()\nstart_date = end_date - timedelta(days=90)\nsamsung_stock = yf.download(\'005930.KS\', start=start_date, end=end_date)\n\n# 그래프 그리기\nfig = go.Figure()\n\n# 주가 선 그래프 추가\nfig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock[\'Close\'], mode=\'lines\', name=\'Close Price\'))\n\n# 그래프 아래를 투

In [37]:
# filename: samsung_stock_price.py
import yfinance as yf
import plotly.graph_objects as go
from datetime import datetime, timedelta

# 삼성전자 주식 데이터 다운로드
end_date = datetime.now()
start_date = end_date - timedelta(days=90)
samsung_stock = yf.download('005930.KS', start=start_date, end=end_date)

# 그래프 그리기
fig = go.Figure()

# 주식 가격 선 그래프 추가
fig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock['Close'].iloc[:,0], mode='lines', name='Close Price'))

# 그래프 아래를 투명한 녹색으로 채우기
fig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock['Close'].iloc[:,0], fill='tozeroy', mode='none', fillcolor='rgba(0, 255, 0, 0.3)'))

# y축 최소값 설정
fig.update_yaxes(range=[samsung_stock['Close'].iloc[:,0].min() * 0.95, samsung_stock['Close'].iloc[:,0].max() * 1.05])

# 레이아웃 설정
fig.update_layout(title='삼성전자 주식 가격 (지난 3개월)',
                  xaxis_title='날짜',
                  yaxis_title='가격 (KRW)',
                  width=800,
                  height=400)

# 그래프 저장
fig.write_image("samsung_stock_price.png")

print("그래프가 samsung_stock_price.png로 저장되었습니다.")

[*********************100%***********************]  1 of 1 completed

그래프가 samsung_stock_price.png로 저장되었습니다.


## 예제 15.6 RAG 에이전트 클래스를 사용한 작업 실행

In [26]:
%pip install --upgrade wandb protobuf -qqq

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Autogen 0.4 버전 부터 RetrieveAssistantAgent, RetrieveUserProxyAgent 없어짐
# Autogen 0.4 버전 Agents, Memory and RAG 참고
# https://microsoft.github.io/autogen/dev//user-guide/agentchat-user-guide/tutorial/agents.html
# https://microsoft.github.io/autogen/dev//user-guide/agentchat-user-guide/memory.html

import autogen
from autogen.agentchat.contrib.retrieve_assistant_agent import RetrieveAssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

In [8]:

assistant = RetrieveAssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config=llm_config,
)


/var/folders/sk/72429_w95sz049jdbs5_8p3n68w8h9/T/ipykernel_4340/200380815.py:1: DeprecationWarning: The RetrieveAssistantAgent is deprecated. Please use the AssistantAgent instead.
  assistant = RetrieveAssistantAgent(


In [ ]:

ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    retrieve_config={
        "task": "qa",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
        "collection_name": "default-sentence-transformers",        
    },
    code_execution_config = {"use_docker": False}
)

assistant.reset()
ragproxyagent.initiate_chat(assistant, problem="AutoGen이 뭐야?")

# assistant (to ragproxyagent):
# AutoGen은 여러 에이전트가 상호 대화하여 작업을 해결할 수 있는 LLM(Large Language Model) 애플리케이션 개발을 가능하게 하는 프레임워크입니다. AutoGen 에이전트는 사용자 정의 가능하며, 대화 가능하고, 인간 참여를 원활하게 허용합니다. LLM, 인간 입력, 도구의 조합을 사용하는 다양한 모드에서 작동할 수 있습니다.

In [5]:
%pip install langchain-experimental -qqq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Define a tool that searches the web for information.
async def web_search(query: str) -> str:
    """Find information on the web"""
    return "AutoGen is a programming framework for building multi-agent applications."


# Create an agent that uses the OpenAI GPT-4o model.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    # api_key="YOUR_API_KEY",
)
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    tools=[web_search],
    system_message="Use tools to solve tasks.",
)

async def assistant_run() -> None:
    response = await agent.on_messages(
        [TextMessage(content="Find information on AutoGen", source="user")],
        cancellation_token=CancellationToken(),
    )
    print(response.inner_messages)
    print(response.chat_message)


# Use asyncio.run(assistant_run()) when running in a script.
await assistant_run()


[ToolCallRequestEvent(source='assistant', models_usage=RequestUsage(prompt_tokens=61, completion_tokens=16), metadata={}, content=[FunctionCall(id='call_gDQMGFkQwSA7Dde9kCrBoUqK', arguments='{"query":"AutoGen"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='assistant', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='AutoGen is a programming framework for building multi-agent applications.', name='web_search', call_id='call_gDQMGFkQwSA7Dde9kCrBoUqK', is_error=False)], type='ToolCallExecutionEvent')]
source='assistant' models_usage=None metadata={} content='AutoGen is a programming framework for building multi-agent applications.' type='ToolCallSummaryMessage'


In [ ]:
import asyncio
import pandas as pd
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from autogen_ext.tools.langchain import LangChainToolAdapter
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken


async def main() -> None:
    df = pd.read_csv("https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv")  # type: ignore
    tool = LangChainToolAdapter(PythonAstREPLTool(locals={"df": df}))
    model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
    agent = AssistantAgent(
        "assistant",
        tools=[tool],
        model_client=model_client,
        system_message="Use the `df` variable to access the dataset.",
    )
    await Console(
        agent.on_messages_stream(
            [TextMessage(content="What's the average age of the passengers?", source="user")], CancellationToken()
        )
    )

await main()

# asyncio.run(main())

---------- assistant ----------
[FunctionCall(id='call_WoCzpPOMlFcdhgTKQm1ekT1g', arguments='{"query":"df[\'Age\'].mean()"}', name='python_repl_ast')]
---------- assistant ----------
[FunctionExecutionResult(content='29.69911764705882', name='python_repl_ast', call_id='call_WoCzpPOMlFcdhgTKQm1ekT1g', is_error=False)]
---------- assistant ----------
29.69911764705882


In [3]:
%pip install -U pydantic pydantic_core -qqq
%pip install -U langchain-openai -qqq
# %pip install pydantic==2.7.1 pydantic_core==2.18.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import requests

from IPython.display import display, Markdown
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA


url = "https://raw.githubusercontent.com/microsoft/autogen/main/README.md"

response = requests.get(url)

# Markdown 렌더링
# display(Markdown(response.text))

# Markdown 문자열 선언
md_text = response.text

# Markdown 문자열을 텍스트로 변환
def parse_markdown(md_string):
    """Markdown에서 텍스트만 추출"""
    return md_string.replace("#", "").replace("**", "").strip()

parsed_text = parse_markdown(md_text)

# 문서 로딩 및 임베딩
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_text(parsed_text)
vectorstore = FAISS.from_texts(documents, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# OpenAI GPT-4o-mini를 사용한 RAG 질의 응답
llm = ChatOpenAI(model_name="gpt-4o-mini")
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

# 예제 질문
query = "Autoget이 뭐야?"
response = qa_chain.run(query)
print("답변:", response)


답변: AutoGen은 Microsoft에서 개발한 도구로, 사용자들이 기여하고 협업할 수 있는 생태계를 제공합니다. 이 도구는 다양한 기능과 문서화, 블로그 튜토리얼 등을 통해 사용자들이 쉽게 접근하고 활용할 수 있도록 돕습니다. 또한, 커뮤니티와의 소통을 위해 Discord 서버와 GitHub Discussions를 운영하고 있습니다. AutoGen에 대한 더 자세한 정보는 [문서](https://microsoft.github.io/autogen/)를 참고하시면 됩니다.


In [ ]:
import asyncio
import sqlite3

import requests
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI
from sqlalchemy import Engine, create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db() -> Engine:
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text
    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


async def main() -> None:
    # Create the engine and database wrapper.
    engine = get_engine_for_chinook_db()
    db = SQLDatabase(engine)

    # Create the toolkit.
    llm = ChatOpenAI(temperature=0)
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)

    # Create the LangChain tool adapter for every tool in the toolkit.
    tools = [LangChainToolAdapter(tool) for tool in toolkit.get_tools()]

    # Create the chat completion client.
    model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

    # Create the assistant agent.
    agent = AssistantAgent(
        "assistant",
        model_client=model_client,
        tools=tools,  # type: ignore
        model_client_stream=True,
        system_message="Respond with 'TERMINATE' if the task is completed.",
    )

    # Create termination condition.
    termination = TextMentionTermination("TERMINATE")

    # Create a round-robin group chat to iterate the single agent over multiple steps.
    chat = RoundRobinGroupChat([agent], termination_condition=termination)

    # Run the chat.
    await Console(chat.run_stream(task="Show some tables in the database"))


await main()

# if __name__ == "__main__":
#     asyncio.run(main())

---------- user ----------
Show some tables in the database
---------- assistant ----------
[FunctionCall(id='call_bu4rCdq1hrnFj8sXPB27ZF4d', arguments='{}', name='sql_db_list_tables')]
---------- assistant ----------
[FunctionExecutionResult(content='Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', name='sql_db_list_tables', call_id='call_bu4rCdq1hrnFj8sXPB27ZF4d', is_error=False)]
---------- assistant ----------
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
---------- assistant ----------
Here are some tables in the database:

1. Album
2. Artist
3. Customer
4. Employee
5. Genre
6. Invoice
7. InvoiceLine
8. MediaType
9. Playlist
10. PlaylistTrack
11. Track

If you need information about a specific table or wish to perform any actions, please let me know!
---------- assistant ----------
TERMINATE


In [4]:
# 필요한 모듈 가져오기
from langchain_experimental.tools import PythonAstREPLTool
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI

# Python 코드 실행을 위한 REPL 도구 초기화
python_repl = PythonAstREPLTool()

# OpenAI의 GPT-4 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 에이전트 초기화
# ZERO_SHOT_REACT_DESCRIPTION 타입의 에이전트를 사용하며, verbose=True로 설정하여 상세 출력을 얻습니다.
agent = initialize_agent(
    [python_repl],  # 사용할 도구 리스트
    llm,  # 사용할 언어 모델
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # 에이전트 타입
    verbose=True  # 상세 출력 설정
)

# 에이전트에게 작업 지시 및 실행
result = agent.run("""
1부터 10까지의 숫자 중 짝수만 출력하는 Python 코드를 작성하고 실행해주세요.
그리고 그 결과를 설명해주세요.
""")

# 결과 출력
print(result)


<frozen abc>:106: RuntimeWarning: coroutine 'main' was never awaited
/var/folders/sk/72429_w95sz049jdbs5_8p3n68w8h9/T/ipykernel_48046/2431052180.py:10: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
/var/folders/sk/72429_w95sz049jdbs5_8p3n68w8h9/T/ipykernel_48046/2431052180.py:14: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph document



> Entering new AgentExecutor chain...
I need to write a Python code that generates even numbers from 1 to 10 and then execute it to see the results. 

Action: python_repl_ast  
Action Input: `even_numbers = [num for num in range(1, 11) if num % 2 == 0]; even_numbers`  
Observation: [2, 4, 6, 8, 10]
Thought:I have generated a list of even numbers from 1 to 10 using a list comprehension. The condition `num % 2 == 0` filters out the even numbers. 

Final Answer: The even numbers from 1 to 10 are [2, 4, 6, 8, 10].

> Finished chain.
The even numbers from 1 to 10 are [2, 4, 6, 8, 10].


## 예제 15.7 외부 정보를 활용하지 못하는 기본 에이전트의 답변

In [ ]:
assistant.reset()
userproxyagent = autogen.UserProxyAgent(
    name="userproxyagent",
)
userproxyagent.initiate_chat(assistant, message="Autogen이 뭐야?")

# assistant (to userproxyagent):
# "Autogen"은 자동 생성을 의미하는 용어로, 주로 컴퓨터 프로그래밍에서 사용됩니다. 이는 코드, 문서, 또는 다른 데이터를 자동으로 생성하는 프로세스를 가리킵니다. 이는 반복적인 작업을 줄이고, 효율성을 높이며, 오류를 줄일 수 있습니다. 특정 컨텍스트에 따라 "Autogen"의 정확한 의미는 다를 수 있습니다.

## 예제 15.8 OpenAI 임베딩 모델을 사용하도록 설정하기

In [ ]:
# Not working in Autogen v0.4

from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
                api_key=openai_api_key,
                model_name="text-embedding-3-small"
            )

ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
        "embedding_function": openai_ef,
        "collection_name": "openai-embedding-3",
    },
    max_consecutive_auto_reply=1,
    code_execution_config={
        "use_docker": False
    }
)

assistant.reset()
ragproxyagent.initiate_chat(assistant, problem="Autogen이 뭐야?")

# assistant (to ragproxyagent):
# AutoGen은 여러 에이전트가 상호 대화하여 작업을 해결할 수 있는 LLM(Large Language Model) 애플리케이션 개발을 가능하게 하는 프레임워크입니다. AutoGen 에이전트는 사용자 정의 가능하며, 대화 가능하고, 인간 참여를 원활하게 허용합니다. LLM, 인간 입력, 도구의 조합을 사용하는 다양한 모드에서 작동할 수 있습니다.

ragproxyagent (to assistant):

Autogen이 뭐야?

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Autogen은 일반적으로 "자동 생성"을 의미하는 용어로, 다양한 분야에서 사용될 수 있습니다. 예를 들어, 소프트웨어 개발에서는 코드, 문서, 테스트 케이스 등을 자동으로 생성하는 도구나 프로세스를 지칭할 수 있습니다. 

특히, 인공지능 및 머신러닝 분야에서는 데이터나 모델을 자동으로 생성하는 기술을 의미할 수 있습니다. 예를 들어, 자연어 처리(NLP)에서는 텍스트를 자동으로 생성하는 모델이 있을 수 있습니다.

어떤 특정한 맥락에서 "Autogen"이라는 용어를 사용하고 있는지에 따라 그 의미가 달라질 수 있으니, 더 구체적인 정보가 필요하다면 추가적인 설명을 부탁드립니다.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):



--------------------------------------------------------------------------------
assistant (to ragproxyagent):

안녕하세요! 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (243e09c2-13bd-48e9-8359-ede7613a2ed5): Maximum number of consecutive auto-replies reached


ChatResult(chat_id=None, chat_history=[{'content': 'Autogen이 뭐야?', 'role': 'assistant', 'name': 'ragproxyagent'}, {'content': 'Autogen은 일반적으로 "자동 생성"을 의미하는 용어로, 다양한 분야에서 사용될 수 있습니다. 예를 들어, 소프트웨어 개발에서는 코드, 문서, 테스트 케이스 등을 자동으로 생성하는 도구나 프로세스를 지칭할 수 있습니다. \n\n특히, 인공지능 및 머신러닝 분야에서는 데이터나 모델을 자동으로 생성하는 기술을 의미할 수 있습니다. 예를 들어, 자연어 처리(NLP)에서는 텍스트를 자동으로 생성하는 모델이 있을 수 있습니다.\n\n어떤 특정한 맥락에서 "Autogen"이라는 용어를 사용하고 있는지에 따라 그 의미가 달라질 수 있으니, 더 구체적인 정보가 필요하다면 추가적인 설명을 부탁드립니다.', 'role': 'user', 'name': 'assistant'}, {'content': '', 'role': 'assistant', 'name': 'ragproxyagent'}, {'content': '안녕하세요! 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요.', 'role': 'user', 'name': 'assistant'}], summary='안녕하세요! 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요.', cost={'usage_including_cached_inference': {'total_cost': 0.00014639999999999998, 'gpt-4o-mini-2024-07-18': {'cost': 0.00014639999999999998, 'prompt_tokens': 228, 'completion_tokens': 187, 'total_tokens': 415}}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_in

## 예제 15.9 대화에 참여할 에이전트

In [12]:
def termination_msg(x):
    return isinstance(x, dict) and "TERMINATE" == str(x.get("content", ""))[-9:].upper()

# RAG를 사용하지 않는 사용자 역할 에이전트
user = autogen.UserProxyAgent(
    name="Admin",
    is_termination_msg=termination_msg,
    human_input_mode="NEVER",
    system_message="The boss who ask questions and give tasks.",
    code_execution_config=False,
    default_auto_reply="Reply `TERMINATE` if the task is done.",
)
# RAG를 사용하는 사용자 역할 에이전트
user_rag = RetrieveUserProxyAgent(
    name="Admin_RAG",
    is_termination_msg=termination_msg,
    system_message="Assistant who has extra content retrieval power for solving difficult problems.",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config=False,
    retrieve_config={
        "task": "code",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/samples/apps/autogen-studio/README.md",
        "chunk_token_size": 1000,
        "collection_name": "groupchat-rag",
    }
)
# 프로그래머 역할의 에이전트
coder = AssistantAgent(
    name="Senior_Python_Engineer",
    is_termination_msg=termination_msg,
    system_message="You are a senior python engineer. Reply `TERMINATE` in the end when everything is done.",
    llm_config=llm_config,
)
# 프로덕트 매니저 역할의 에이전트
pm = autogen.AssistantAgent(
    name="Product_Manager",
    is_termination_msg=termination_msg,
    system_message="You are a product manager. Reply `TERMINATE` in the end when everything is done.",
    llm_config=llm_config,
)

PROBLEM = "AutoGen Studio는 무엇이고 AutoGen Studio로 어떤 제품을 만들 수 있을까?"

## 예제 15.10 RAG 사용 여부에 따른 2개의 그룹챗 정의 및 실행

In [13]:
def _reset_agents():
    user.reset()
    user_rag.reset()
    coder.reset()
    pm.reset()

def rag_chat():
    _reset_agents()
    groupchat = autogen.GroupChat(
        agents=[user_rag, coder, pm],
        messages=[], max_round=12, speaker_selection_method="round_robin"
    )
    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

    user_rag.initiate_chat(
        manager,
        problem=PROBLEM,
    )

def norag_chat():
    _reset_agents()
    groupchat = autogen.GroupChat(
        agents=[user, coder, pm],
        messages=[],
        max_round=12,
        speaker_selection_method="auto",
        allow_repeat_speaker=False,
    )
    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

    user.initiate_chat(
        manager,
        message=PROBLEM,
    )

## 예제 15.11 2개의 그룹챗을 실행한 결과 비교

In [ ]:
# Not working. RetrieveUserProxyAgent class is deplecated on AutoGen v0.4.

norag_chat()
# AutoGen Studio는 자동화된 코드 생성 도구입니다. 이 도구를 사용하면 개발자들이 더 빠르게, 더 효율적으로 코드를 작성할 수 있습니다.
# AutoGen Studio를 사용하면 다양한 유형의 소프트웨어 제품을 만들 수 있습니다. 예를 들어, 웹 애플리케이션, 모바일 애플리케이션, 데스크톱 애플리케이션, API, 데이터베이스 등을 만들 수 있습니다.
# ...

Admin (to chat_manager):

AutoGen Studio는 무엇이고 AutoGen Studio로 어떤 제품을 만들 수 있을까?

--------------------------------------------------------------------------------

Next speaker: Product_Manager

Product_Manager (to chat_manager):

AutoGen Studio는 자동화된 코드 생성 및 개발 도구로, 사용자가 소프트웨어 애플리케이션을 보다 쉽게 만들 수 있도록 돕는 플랫폼입니다. 이 도구는 다양한 프로그래밍 언어와 프레임워크를 지원하며, 사용자가 요구하는 기능에 맞춰 코드를 자동으로 생성해 줍니다.

AutoGen Studio를 사용하여 만들 수 있는 제품의 예시는 다음과 같습니다:

1. **웹 애플리케이션**: 사용자 인터페이스와 백엔드 로직을 자동으로 생성하여 빠르게 웹 애플리케이션을 개발할 수 있습니다.
2. **모바일 애플리케이션**: iOS 및 Android 플랫폼을 위한 모바일 앱을 손쉽게 만들 수 있습니다.
3. **API 서비스**: RESTful API 또는 GraphQL API를 자동으로 생성하여 다른 애플리케이션과의 통신을 용이하게 할 수 있습니다.
4. **데이터베이스 관리 시스템**: 데이터 모델을 정의하고, CRUD(Create, Read, Update, Delete) 기능을 자동으로 생성하여 데이터베이스와의 상호작용을 간소화할 수 있습니다.
5. **프로토타입 및 MVP**: 아이디어를 신속하게 프로토타입으로 구현하여 시장 반응을 테스트할 수 있습니다.

이러한 제품들은 AutoGen Studio의 기능을 활용하여 개발 시간을 단축하고, 코드 품질을 향상시키며, 개발자들이 더 창의적인 작업에 집중할 수 있도록 도와줍니다.

TERMINATE

------------------------------------------------------------------

In [15]:
rag_chat()
# AutoGen Studio는 AutoGen 프레임워크를 기반으로 한 AI 앱입니다. 이 앱은 AI 에이전트를 빠르게 프로토타입화하고, 스킬을 향상시키고, 워크플로우로 구성하고, 작업을 완료하기 위해 그들과 상호 작용하는 데 도움을 줍니다. 이 앱은 GitHub의 [microsoft/autogen](https://github.com/microsoft/autogen/tree/main/samples/apps/autogen-studio)에서 코드를 찾을 수 있습니다.
# AutoGen Studio를 사용하면 다음과 같은 기능을 수행할 수 있습니다:
# - 에이전트를 구축/구성하고, 그들의 구성(예: 스킬, 온도, 모델, 에이전트 시스템 메시지, 모델 등)을 수정하고, 워크플로우로 구성합니다.
# ...

Admin_RAG (to chat_manager):

AutoGen Studio는 무엇이고 AutoGen Studio로 어떤 제품을 만들 수 있을까?

--------------------------------------------------------------------------------

Next speaker: Senior_Python_Engineer

Senior_Python_Engineer (to chat_manager):

AutoGen Studio는 자동화된 코드 생성 및 개발 도구로, 주로 소프트웨어 개발 프로세스를 간소화하고 가속화하는 데 사용됩니다. 이 도구는 다양한 프로그래밍 언어와 프레임워크를 지원하며, 개발자가 반복적인 작업을 줄이고 더 빠르게 프로토타입을 만들 수 있도록 도와줍니다.

AutoGen Studio를 사용하여 만들 수 있는 제품의 예는 다음과 같습니다:

1. **웹 애플리케이션**: 사용자 인터페이스와 백엔드 로직을 자동으로 생성하여 빠르게 웹 애플리케이션을 개발할 수 있습니다.

2. **모바일 애플리케이션**: iOS 및 Android 플랫폼을 위한 모바일 앱을 신속하게 생성할 수 있습니다.

3. **API 서비스**: RESTful API 또는 GraphQL API를 자동으로 생성하여 데이터 통신을 간소화할 수 있습니다.

4. **데이터베이스 스키마**: 데이터베이스 모델을 자동으로 생성하고, CRUD(Create, Read, Update, Delete) 작업을 위한 코드를 생성할 수 있습니다.

5. **테스트 스크립트**: 자동화된 테스트 스크립트를 생성하여 소프트웨어의 품질을 보장할 수 있습니다.

6. **프로토타입**: 아이디어를 신속하게 프로토타입으로 변환하여 피드백을 받을 수 있습니다.

이러한 제품들은 개발 시간을 단축시키고, 코드의 일관성을 높이며, 개발자들이 더 창의적인 작업에 집중할 수 있도록 도와줍니다. 

TERMINATE

---------------------------------

## 예제 15.12 실습 준비하기

In [ ]:
import os
import re
import time
from typing import Any, Callable, Dict, List, Optional, Tuple, Type, Union

import matplotlib.pyplot as plt
import PIL
import requests
from openai import OpenAI
from PIL import Image

from autogen import Agent, AssistantAgent, ConversableAgent, UserProxyAgent
from autogen.agentchat.contrib.img_utils import _to_pil, get_image_data
from autogen.agentchat.contrib.multimodal_conversable_agent import MultimodalConversableAgent

config_list_4o = autogen.config_list_from_json(
    "OAI_CONFIG_LIST.json",
    filter_dict={
        "model": ["gpt-4o"],
    },
)

config_list_dalle = autogen.config_list_from_json(
    "OAI_CONFIG_LIST.json",
    filter_dict={
        "model": ["dall-e-3"],
    },
)

## 예제 15.13 DALLEAgent 정의

In [ ]:
def dalle_call(client, prompt, model="dall-e-3", size="1024x1024", quality="standard", n=1) -> str:
    response = client.images.generate(
        model=model,
        prompt=prompt,
        size=size,
        quality=quality,
        n=n,
    )
    image_url = response.data[0].url
    img_data = get_image_data(image_url)
    return img_data

class DALLEAgent(ConversableAgent):
    def __init__(self, name, llm_config: dict, **kwargs):
        super().__init__(name, llm_config=llm_config, **kwargs)

        try:
            config_list = llm_config["config_list"]
            api_key = config_list[0]["api_key"]
        except Exception as e:
            print("Unable to fetch API Key, because", e)
            api_key = os.getenv("OPENAI_API_KEY")
        self.client = OpenAI(api_key=api_key)
        self.register_reply([Agent, None], DALLEAgent.generate_dalle_reply)

    def generate_dalle_reply(self, messages, sender, config):
        client = self.client if config is None else config
        if client is None:
            return False, None
        if messages is None:
            messages = self._oai_messages[sender]

        prompt = messages[-1]["content"]
        img_data = dalle_call(client=self.client, prompt=prompt)
        plt.imshow(_to_pil(img_data))
        plt.axis("off")
        plt.show()
        return True, 'result.jpg'

## 예제 15.14 이미지 생성 에이전트 실행

In [ ]:
painter = DALLEAgent(name="Painter", llm_config={"config_list": config_list_dalle})

user_proxy = UserProxyAgent(
    name="User_proxy", system_message="A human admin.", human_input_mode="NEVER", max_consecutive_auto_reply=0
)

# 이미지 생성 작업 실행하기
user_proxy.initiate_chat(
    painter,
    message="갈색의 털을 가진 귀여운 강아지를 그려줘",
)

## 예제 15.15 이미지를 입력으로 받을 수 있는 GPT-4o 에이전트 생성

In [ ]:
image_agent = MultimodalConversableAgent(
    name="image-explainer",
    system_message="Explane input image for painter to create similar image.",
    max_consecutive_auto_reply=10,
    llm_config={"config_list": config_list_4o, "temperature": 0.5, "max_tokens": 1500},
)

user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin.",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False
)

groupchat = autogen.GroupChat(agents=[user_proxy, image_agent, painter], messages=[], max_round=12)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

## 예제 15.16 유사한 이미지를 생성하도록 에이전트 실행

In [ ]:
user_proxy.initiate_chat(
    manager,
    message=f"""아래 이미지랑 비슷한 이미지를 만들어줘.
<img https://th.bing.com/th/id/R.422068ce8af4e15b0634fe2540adea7a?rik=y4OcXBE%2fqutDOw&pid=ImgRaw&r=0>.""",
)

## 예제 15.18 멀티 모달 에이전트에 텍스트로 명령

In [ ]:
user_proxy.initiate_chat(
    manager,
    message="갈색의 털을 가진 귀여운 강아지를 그려줘",
)